# Vectorising Text: TF-IDF

**Computational Text Analysis Pack · Module 06 — Clustering Documents**

*Technique:* Document–term matrices; term weighting and what min_df does

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/05-unsupervised-structure/06a_vectorising_text.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Every method from here on needs documents as **numbers**. The standard move is
one row per document, one column per word, and a number in each cell.

That matrix is the bridge between everything you learned in the statistics course
and everything you know about text. Once documents are rows of numbers, k-means,
PCA and classifiers apply unchanged.

Two weightings matter: **counts** and **TF-IDF**. The second is almost always the
one you want, and this notebook shows why by comparing them.

## The tooling

**scikit-learn** — `TfidfVectorizer` for the document-term matrix, `KMeans` and
`TruncatedSVD` for structure. **scipy**'s `cluster.hierarchy` does the
dendrograms. **gensim** is the alternative once the matrix stops fitting in
memory.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

LANGUAGE = "English"        # ← change this
MIN_DF = 5                  # ← a word must appear in at least this many documents
MAX_FEATURES = 3000

sub = df[df["language"] == LANGUAGE].reset_index(drop=True)
print(f"{len(sub):,} documents")

count_vec = CountVectorizer(token_pattern=r"\w+", min_df=MIN_DF,
                            max_features=MAX_FEATURES, lowercase=True)
X_counts = count_vec.fit_transform(sub["text"])
print(f"count matrix : {X_counts.shape[0]:,} documents × {X_counts.shape[1]:,} words")
print(f"density      : {X_counts.nnz / (X_counts.shape[0] * X_counts.shape[1]):.2%} "
      "of cells are non-zero")

Note `token_pattern=r"\w+"`. scikit-learn's default is `(?u)\b\w\w+\b`, which
is Unicode-aware but drops every two-letter word — and in Yoruba *ni* and *ti*,
two of the commonest words in the language, are two letters. Setting the pattern
explicitly is the 03b lesson applied one layer up.

In [ ]:
# What the matrix holds, for one document.
doc_id = 0
vocab = np.array(count_vec.get_feature_names_out())
row = X_counts[doc_id].toarray().ravel()
top = np.argsort(row)[::-1][:12]

print("headline:", sub.iloc[doc_id]["headline"][:90])
print()
print(f"{'word':<18}{'count'}")
for i in top:
    if row[i]:
        print(f"{vocab[i]:<18}{row[i]}")

## Why counts are the wrong weighting

The top of that list is *the*, *of*, *to* — for every document. Counts make all
documents look alike, because the commonest words dominate every row.

TF-IDF fixes it with one idea: **divide by how many documents the word appears
in**. A word in every document carries no information about any of them; a word
in a few carries a lot.

In [ ]:
tfidf_vec = TfidfVectorizer(token_pattern=r"\w+", min_df=MIN_DF,
                            max_features=MAX_FEATURES, sublinear_tf=True)
X = tfidf_vec.fit_transform(sub["text"])
vocab_t = np.array(tfidf_vec.get_feature_names_out())

row = X[doc_id].toarray().ravel()
top = np.argsort(row)[::-1][:12]
print("Same document, TF-IDF weighting:")
print(f"{'word':<18}{'tf-idf'}")
for i in top:
    print(f"{vocab_t[i]:<18}{row[i]:.3f}")
print()
print("Now the top words are about the document rather than about English.")

In [ ]:
# The IDF component, which is where the work happens.
idf = tfidf_vec.idf_
order = np.argsort(idf)
print("Lowest IDF — in nearly every document, therefore nearly worthless:")
print("  " + ", ".join(vocab_t[order[:12]]))
print()
print("Highest IDF — rare, therefore highly distinctive when present:")
print("  " + ", ".join(vocab_t[order[-12:]]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(idf, bins=50)
ax.set_xlabel("inverse document frequency")
ax.set_ylabel("words")
ax.set_title("Most of the vocabulary is rare — Zipf again, seen from a new angle")
plt.tight_layout(); plt.show()

### 🔧 Your turn 1

Set `MIN_DF` to 1, then 50, and watch the vocabulary size change.

At 1 you keep every word including typos and one-off names — thousands of columns
that are almost all zero. At 50 you keep only words common enough to be shared,
and may have deleted the distinctive terms your question depends on.

## Sparsity, and why it is fine

Look back at the density figure: well under 1% of the matrix is non-zero. A
document uses a hundred distinct words out of a three-thousand-word vocabulary.

scikit-learn stores this as a **sparse matrix** — only the non-zero cells. That
is why an 8,000 × 3,000 matrix takes megabytes rather than gigabytes. Calling
`.toarray()` on a large one is the fastest way to run a notebook out of memory.

In [ ]:
dense_mb = X.shape[0] * X.shape[1] * 8 / 1024 / 1024
print(f"sparse storage : {X.data.nbytes / 1024 / 1024:>8.1f} MB")
print(f"if made dense  : {dense_mb:>8.1f} MB")
print()
print("Keep it sparse. Every method in this pack accepts a sparse matrix.")

### 🔧 Your turn 2

Build a TF-IDF matrix for Yoruba or Hausa instead of English.

Check the vocabulary size against English at the same `min_df`. Morphologically
richer languages produce far more distinct forms — the fat Zipf tail from 02b,
now costing you columns.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** `min_df=1` typically triples or quadruples the vocabulary,
almost entirely with words appearing once. They add columns, add noise, and
cannot support any conclusion. `min_df=50` gives a compact matrix that clusters
cleanly and may have removed the very terms that distinguish your subgroups.
Between 3 and 10 is the usual working range; report the value.

**Your turn 2.** Yoruba and Hausa produce noticeably more types for the same
number of documents, because inflected forms are separate types with no
lemmatiser to merge them. Practical consequence: at a fixed `max_features` you
cover proportionally less of an African-language corpus than of an English one.
If you compare model performance across languages, that difference is part of the
explanation and belongs in your discussion.

</details>

## Where this stops

Documents are numbers now. Next: **06b — k-means**, which groups them.